## 1. Prepare the county data

In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
dp03 = pd.read_csv("DP03_EconomicCharacteristics.csv")
dp04 = pd.read_csv("DP04_HousingCharacteristics.csv")

In [ ]:
dp03.columns = dp03.columns.str.replace("!!", "_", regex = False)
dp04.columns = dp04.columns.str.replace("!!", "_", regex = False)
display(dp03.head())
display(dp04.head())

In [ ]:
label03 = dp03.columns[0]
label04 = dp04.columns[0]
dp03[dp03[label03].astype(str).str.contains("median household income", case=False, na=False)]

In [ ]:
rent_start = dp04.index[
    dp04[label04].astype(str).str.strip().str.casefold().eq("gross rent")
][0]
rent_section = dp04.loc[rent_start + 1:]
rent_row = rent_section[
    rent_section[label04].astype(str).str.strip().str.casefold().eq("median (dollars)")
]
rent_row

In [ ]:
income_row = dp03[dp03[label03].astype(str).str.contains("median household income", case=False, na=False)]
income = (income_row.filter(regex="_Estimate$").T.reset_index())
rent = (rent_row.filter(regex="_Estimate$").T.reset_index())


In [ ]:
income.columns = ["County", "Median_Household_Income"]
rent.columns = ["County", "Median_Gross_Rent"]
display(income.head())
display(rent.head())

In [ ]:
housing = pd.merge(income, rent, on = "County", how = "inner")
display(housing.head())

In [ ]:
housing["Median_Household_Income"] = pd.to_numeric(housing["Median_Household_Income"].astype(str).str.replace(",", ""), errors = "coerce")
housing["Median_Gross_Rent"] = pd.to_numeric(housing["Median_Gross_Rent"].astype(str).str.replace(",", ""), errors = "coerce")

In [ ]:
housing["Annual_Rent"] = housing["Median_Gross_Rent"] * 12
display(housing.head())

In [ ]:
housing["Rent_to_Income_Ratio"] = housing["Annual_Rent"]/housing["Median_Household_Income"] * 100
display(housing.head())

In [ ]:
housing.sort_values("Rent_to_Income_Ratio", ascending=False).head(10)

In [ ]:
housing.shape
housing.dtypes
housing.describe()

In [ ]:
housing["County"] = housing["County"].str.replace("_Estimate", "", regex=False)
housing = housing[housing["County"] != "Georgia"].copy()
housing.describe()

In [ ]:
housing[housing["County"] == "Georgia"]

In [ ]:
housing["County"].eq("Georgia").any()

## 2. Explore the data

In [ ]:
plt.scatter(housing["Median_Household_Income"], housing["Median_Gross_Rent"])
plt.xlabel("Median Household Income ($)")
plt.ylabel("Median Gross Rent ($)")
plt.title("Household Income and Rent Across Georgia Counties")
plt.xlim(0, 150000)
plt.ylim(0, 2500)
plt.show()

In [ ]:
correlation = housing["Median_Household_Income"].corr(housing["Median_Gross_Rent"])
print(correlation)

In [ ]:
housing.sort_values("Rent_to_Income_Ratio",ascending=False)[["County", "Rent_to_Income_Ratio"]].head(10)

In [ ]:
housing.sort_values("Rent_to_Income_Ratio",ascending=True)[["County", "Rent_to_Income_Ratio"]].head(10)

In [ ]:
plt.scatter(housing["Median_Household_Income"], housing["Rent_to_Income_Ratio"])
plt.xlabel("Median Household Income ($)")
plt.ylabel("Rent To Income Ratio (%)")
plt.title("Income to Rent Affordability Across Georgia Counties")

In [ ]:
correlation = housing["Median_Household_Income"].corr(housing["Rent_to_Income_Ratio"])
print(correlation)

In [ ]:
housing["Income_Thousands"] = (housing["Median_Household_Income"] / 1000)

## 3. Run the original regression models

In [ ]:
%pip install statsmodels

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
model1 = smf.ols("Median_Gross_Rent ~ Income_Thousands", data=housing).fit()
print(model1.summary())

In [ ]:
plt.scatter(housing["Income_Thousands"], housing["Median_Gross_Rent"])
line_data = housing.sort_values("Income_Thousands")
y_hat = (
    model1.params["Intercept"]
    + model1.params["Income_Thousands"] * line_data["Income_Thousands"]
)
plt.plot(line_data["Income_Thousands"], y_hat, color="red")
plt.xlabel("Median Household Income ($1,000s)")
plt.ylabel("Median Gross Rent ($)")
plt.title("Median Household Income and Gross Rent Across Georgia Counties")
plt.show()

In [ ]:
dp03.loc[125:145, [label03,"Georgia_Percent","Appling County, Georgia_Percent"]]

In [ ]:
poverty_row = dp03[
    dp03[label03].astype(str).str.strip().str.casefold().eq("all people")
]
poverty = poverty_row.filter(regex="_Percent$").T.reset_index()
poverty.columns = ["County", "Poverty_Rate"]
poverty.head()

In [ ]:
print(housing.shape)
print(poverty.shape)

In [ ]:
housing["County"] = housing["County"].str.replace("_Estimate", "", regex=False)
poverty["County"] = poverty["County"].str.replace("_Percent", "", regex=False)
poverty["Poverty_Rate"] = pd.to_numeric(poverty["Poverty_Rate"].str.replace("%", "", regex=False), errors="coerce")
housing = housing[housing["County"] != "Georgia"].copy()
poverty = poverty[poverty["County"] != "Georgia"].copy()
print(housing.shape)
print(poverty.shape)

In [ ]:
housing = housing.merge(
    poverty,
    on="County",
    how="left",
    validate="one_to_one"
)
housing.head()

In [ ]:
housing.head()

In [ ]:
model2 = smf.ols("Median_Gross_Rent ~ Income_Thousands + Poverty_Rate", data=housing).fit()
print(model2.summary())

In [ ]:
model3 = smf.ols("Median_Gross_Rent ~ Poverty_Rate", data=housing).fit()
print(model3.summary())

In [ ]:
plt.scatter(housing["Poverty_Rate"], housing["Median_Gross_Rent"])
line_data = housing.sort_values("Poverty_Rate")
x = line_data["Poverty_Rate"]
y_hat = (
    model3.params["Intercept"]
    + model3.params["Poverty_Rate"] * x
)
plt.plot(x, y_hat, color="red")
plt.xlabel("Poverty Rate (%)")
plt.ylabel("Median Gross Rent ($)")
plt.title("Poverty Rates and Gross Rent Across Georgia Counties")
plt.show()

In [ ]:
rental_vacancy_row = dp04[dp04[label04].astype(str).str.contains("rental vacancy rate", case=False, na=False)]
rental_vacancy = (rental_vacancy_row.filter(regex="_Estimate$").T.reset_index())
rental_vacancy.columns = ["County", "Rental_Vacancy_Rate"]
rental_vacancy.head()

In [ ]:
print(rental_vacancy.shape)

In [ ]:
rental_vacancy["County"] = rental_vacancy["County"].str.replace("_Estimate", "", regex=False)
rental_vacancy = rental_vacancy[rental_vacancy["County"] != "Georgia"].copy()
print(rental_vacancy.shape)


In [ ]:
housing.head()

In [ ]:
housing = housing.merge(
    rental_vacancy,
    on="County",
    how="left",
    validate="one_to_one"
)
housing.head()

In [ ]:
housing[["Income_Thousands", "Rental_Vacancy_Rate"]].corr()

In [ ]:
model4 = smf.ols("Median_Gross_Rent ~ Income_Thousands + Rental_Vacancy_Rate", data=housing).fit()
print(model4.summary())

In [ ]:
model5 = smf.ols("Median_Gross_Rent ~ Rental_Vacancy_Rate", data=housing).fit()
print(model5.summary())

In [ ]:
plt.scatter(housing["Rental_Vacancy_Rate"], housing["Median_Gross_Rent"])
line_data = housing.sort_values("Rental_Vacancy_Rate")
y_hat = (
    model5.params["Intercept"]
    + model5.params["Rental_Vacancy_Rate"] * line_data["Rental_Vacancy_Rate"]
)
plt.plot(line_data["Rental_Vacancy_Rate"], y_hat, color="red")
plt.xlabel("Rental Vacancy Rate (%)")
plt.ylabel("Median Gross Rent ($)")
plt.title("Rental Vacancy Rates and Gross Rent Across Georgia Counties")
plt.show()

## 4. Add rent burden and compare models

In [ ]:
def extract_county_row(table, label_column, row_label, suffix, variable_name):
    labels = table[label_column].astype(str).str.strip().str.casefold()
    matches = table.loc[labels.eq(row_label.strip().casefold())]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one row labeled {row_label!r}, found {len(matches)}."
        )
    columns = table.columns[table.columns.str.endswith(f"_{suffix}")]
    values = matches.iloc[0][columns].rename("Value").reset_index()
    values.columns = ["County", variable_name]
    values["County"] = values["County"].str.replace(
        rf"_{suffix}$", "", regex=True
    )
    values[variable_name] = (
        values[variable_name]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    values[variable_name] = pd.to_numeric(values[variable_name], errors="coerce")
    return values
grapi_start = dp04.index[
    dp04[label04].astype(str).str.strip().str.casefold().eq(
        "gross rent as a percentage of household income (grapi)"
    )
][0]
grapi_section = dp04.loc[grapi_start + 1:]
burden_30_34 = extract_county_row(
    grapi_section, label04, "30.0 to 34.9 percent",
    "Percent", "Burden_30_34_Rate"
)
burden_35_plus = extract_county_row(
    grapi_section, label04, "35.0 percent or more",
    "Percent", "Burden_35_Plus_Rate"
)
eligible_renters = extract_county_row(
    grapi_section, label04,
    "Occupied units paying rent (excluding units where GRAPI cannot be computed)",
    "Estimate", "Eligible_Renter_Households"
)
rent_burden = (
    burden_30_34
    .merge(burden_35_plus, on="County", validate="one_to_one")
    .merge(eligible_renters, on="County", validate="one_to_one")
)
rent_burden["Rent_Burden_Rate"] = rent_burden[
    ["Burden_30_34_Rate", "Burden_35_Plus_Rate"]
].sum(axis=1, min_count=2)
statewide_burden = rent_burden.loc[
    rent_burden["County"].eq("Georgia"), "Rent_Burden_Rate"
].iloc[0]
print("Statewide rent-burden estimate: {:.1f}%".format(statewide_burden))
rent_burden = rent_burden[rent_burden["County"] != "Georgia"].copy()
housing = housing.merge(
    rent_burden[["County", "Rent_Burden_Rate", "Eligible_Renter_Households"]],
    on="County",
    how="left",
    validate="one_to_one"
)
print("County rows in housing:", len(housing))
display(housing[["County", "Poverty_Rate", "Rental_Vacancy_Rate",
                 "Rent_Burden_Rate"]].head())

In [ ]:
analysis_variables = [
    "Median_Gross_Rent",
    "Income_Thousands",
    "Poverty_Rate",
    "Rental_Vacancy_Rate",
    "Rent_Burden_Rate",
    "Eligible_Renter_Households"
]
analysis = housing.dropna(subset=analysis_variables).copy()
print("Complete counties used in the expanded models:", len(analysis))
print("Duplicate counties:", analysis["County"].duplicated().sum())
display(
    analysis[
        ["Median_Gross_Rent", "Median_Household_Income", "Poverty_Rate",
         "Rental_Vacancy_Rate", "Rent_Burden_Rate",
         "Rent_to_Income_Ratio"]
    ].describe().T.round(1)
)
print("Unweighted average county rent-burden rate: {:.1f}%".format(
    analysis["Rent_Burden_Rate"].mean()
))
display(
    analysis.sort_values("Rent_Burden_Rate", ascending=False)[
        ["County", "Rent_Burden_Rate", "Median_Gross_Rent", "Income_Thousands"]
    ].head(10).round(1)
)

In [ ]:
analysis["Income_Thousands"] = analysis["Median_Household_Income"] / 1000
model_income_only = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands", data=analysis
).fit()
model_income_poverty = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands + Poverty_Rate", data=analysis
).fit()
model_income_vacancy = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands + Rental_Vacancy_Rate", data=analysis
).fit()
model_full_rent = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands + Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis
).fit()
def compare_models(models):
    rows = []
    for name, model in models.items():
        rows.append({
            "Model": name,
            "N": int(model.nobs),
            "R_squared": model.rsquared,
            "Adjusted_R_squared": model.rsquared_adj,
            "AIC": model.aic,
            "BIC": model.bic
        })
    return pd.DataFrame(rows).set_index("Model")
rent_models = {
    "Income only": model_income_only,
    "Income + poverty": model_income_poverty,
    "Income + vacancy": model_income_vacancy,
    "Income + poverty + vacancy": model_full_rent
}
rent_model_table = compare_models(rent_models)
display(rent_model_table.round(3))
print(model_full_rent.summary())

In [ ]:
income_slope = model_full_rent.params["Income_Thousands"]
income_interval = model_full_rent.conf_int().loc["Income_Thousands"]
print("Adjusted income slope: " + "$" + "{:.2f} per $1,000".format(income_slope))
print("95% confidence interval: " + "$" + "{:.2f} to ".format(income_interval[0])
      + "$" + "{:.2f}".format(income_interval[1]))
print("Full-model R-squared: {:.3f}".format(model_full_rent.rsquared))
print("Full-model adjusted R-squared: {:.3f}".format(model_full_rent.rsquared_adj))

## 5. Check model reliability

In [ ]:
model_full_rent_hc3 = model_full_rent.get_robustcov_results(cov_type="HC3")
print(model_full_rent_hc3.summary())

In [ ]:
from patsy import dmatrix
from statsmodels.stats.outliers_influence import variance_inflation_factor
design = dmatrix(
    "Income_Thousands + Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis,
    return_type="dataframe"
)
vif_table = pd.DataFrame({
    "Variable": design.columns[1:],
    "VIF": [
        variance_inflation_factor(design.values, i)
        for i in range(1, design.shape[1])
    ]
})
display(vif_table.round(2))

In [ ]:
from statsmodels.stats.anova import anova_lm
model_rent_quadratic = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands + I(Income_Thousands ** 2) + "
    "Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis
).fit()
display(compare_models({
    "Full linear model": model_full_rent,
    "Quadratic income model": model_rent_quadratic
}).round(3))
display(anova_lm(model_full_rent, model_rent_quadratic))
model_rent_interaction = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands * Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis
).fit()
display(compare_models({
    "Full linear model": model_full_rent,
    "Income x poverty interaction": model_rent_interaction
}).round(3))
display(anova_lm(model_full_rent, model_rent_interaction))

In [ ]:
import statsmodels.api as sm
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(model_full_rent.fittedvalues, model_full_rent.resid, alpha=0.7)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted median gross rent ($)")
axes[0].set_ylabel("Residual ($)")
axes[0].set_title("Residuals vs. fitted values")
sm.qqplot(model_full_rent.resid, line="45", ax=axes[1])
axes[1].set_title("Q-Q plot of rent-model residuals")
plt.tight_layout()
plt.show()

In [ ]:
influence = model_full_rent.get_influence()
cooks_distance = pd.Series(
    influence.cooks_distance[0],
    index=analysis.index,
    name="Cooks_Distance"
)
cooks_cutoff = 4 / len(analysis)
top_cooks = cooks_distance.nlargest(10)
influence_table = (
    analysis.loc[top_cooks.index,
        ["County", "Income_Thousands", "Poverty_Rate",
         "Rental_Vacancy_Rate", "Median_Gross_Rent"]]
    .assign(Cooks_Distance=top_cooks)
)
print("Reference screening value 4/n = {:.4f}".format(cooks_cutoff))
display(influence_table.round(3))
top_county = cooks_distance.idxmax()
analysis_without_top = analysis.drop(index=top_county)
model_without_top = smf.ols(
    "Median_Gross_Rent ~ Income_Thousands + Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis_without_top
).fit()
sensitivity = pd.DataFrame({
    "Sample": ["All counties", "Without " + analysis.loc[top_county, "County"]],
    "N": [int(model_full_rent.nobs), int(model_without_top.nobs)],
    "Income slope": [
        model_full_rent.params["Income_Thousands"],
        model_without_top.params["Income_Thousands"]
    ],
    "R_squared": [model_full_rent.rsquared, model_without_top.rsquared]
})
display(sensitivity.round(3))

## 6. Model renter affordability

In [ ]:
model_burden_income = smf.ols(
    "Rent_Burden_Rate ~ Income_Thousands", data=analysis
).fit()
model_burden_income_poverty = smf.ols(
    "Rent_Burden_Rate ~ Income_Thousands + Poverty_Rate", data=analysis
).fit()
model_full_burden = smf.ols(
    "Rent_Burden_Rate ~ Income_Thousands + Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis
).fit()
burden_models = {
    "Income only": model_burden_income,
    "Income + poverty": model_burden_income_poverty,
    "Income + poverty + vacancy": model_full_burden
}
burden_model_table = compare_models(burden_models)
display(burden_model_table.round(3))
print(model_full_burden.summary())
print("HC3 robust standard errors:")
print(model_full_burden.get_robustcov_results(cov_type="HC3").summary())

In [ ]:
model_burden_weighted = smf.wls(
    "Rent_Burden_Rate ~ Income_Thousands + Poverty_Rate + Rental_Vacancy_Rate",
    data=analysis,
    weights=analysis["Eligible_Renter_Households"]
).fit(cov_type="HC3")
print(model_burden_weighted.summary())

In [ ]:
plt.scatter(analysis["Income_Thousands"], analysis["Rent_Burden_Rate"], alpha=0.7)
line_data = analysis.sort_values("Income_Thousands")
plt.plot(
    line_data["Income_Thousands"],
    model_burden_income.predict(line_data),
    color="red"
)
plt.xlabel("Median Household Income ($1,000s)")
plt.ylabel("Renter households spending 30%+ on rent (%)")
plt.title("County Income and Renter Cost Burden in Georgia")
plt.grid(alpha=0.2)
plt.show()

In [ ]:
proxy_comparison = analysis[
    ["County", "Rent_to_Income_Ratio", "Rent_Burden_Rate"]
].sort_values("Rent_Burden_Rate", ascending=False)
print("Correlation of the median-based proxy with DP04 rent burden:")
print(proxy_comparison["Rent_to_Income_Ratio"].corr(
    proxy_comparison["Rent_Burden_Rate"]
))
display(proxy_comparison.head(10).round(1))

In [ ]:
from pathlib import Path
Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
rent_model_table.to_csv("results/rent_model_comparison.csv")
burden_model_table.to_csv("results/rent_burden_model_comparison.csv")
analysis.to_csv("results/georgia_county_analysis.csv", index=False)
print("Saved model tables and cleaned county data in the results folder.")